# Construction Materials

This notebook shows how to:
1. Extract construction costs from buildings
2. Use `format_production_list()` method
3. Get maintenance costs
4. Compare costs across buildings

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Extract Construction Costs

Buildings have a `Cost.Costs` property with construction materials:

In [2]:
def extract_construction_costs(building):
    """Extract construction costs from a building."""
   
    cost_list = building.find("Cost.Costs")
    if cost_list:
        return assets.properties.ui_text_cache.format_product_list(cost_list)
    
    return None

# Get a building
# Try to find a production building
building = None
if "Production" in assets.templates.elements:
    building = list(assets.templates["Production"].assets)[0]

if building:
    name = building.text() if building.text else 'N/A'
    print(f"Building: {name}\n")
    
    costs = extract_construction_costs(building)
    
    if costs:
        print("Construction Costs:")
        for ui in costs:
            print(f"  - {ui.value} {ui.text}")
    else:
        print("No construction costs found")
else:
    print("No building found")

Building: Fishing Hut

Construction Costs:
  - 2 Timber


## Extract Maintenance Costs

Buildings also have maintenance costs (ongoing costs):

In [3]:
from assetextractor.parsing.core.attributes import ListAttribute


def extract_maintenance_costs(building):
    """Extract maintenance costs from a building."""
  
    # Check for Maintenance.Maintenances
    maintenance_list = building.find("Maintenance.Maintenances")
    if isinstance(maintenance_list, ListAttribute):
        return assets.properties.ui_text_cache.format_product_list(maintenance_list)

    
    return None

if building:
    maintenance = extract_maintenance_costs(building)
    
    if maintenance:
        print("\nMaintenance Costs:")
        for ui in maintenance:
            print(f"  - {ui.value} {ui.text}")
    else:
        print("\nNo maintenance costs")


Maintenance Costs:
  - 6 Denarii
  - 4 Libertus Workforce


## Export Cost Data to CSV

Export construction costs for further analysis:

In [14]:
import csv
from pathlib import Path

def export_costs_to_csv(template_name, output_file):
    """Export construction costs to CSV."""
    if template_name not in assets.templates.elements:
        print(f"Template '{template_name}' not found")
        return
    
    template = assets.templates[template_name]
    
    output_dir = Path("results/example")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / output_file
    
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['GUID', 'Name', 'Material', 'Amount'])
        
        for building in template.assets:
            name = building.text.values.get(LANGUAGE, 'N/A') if building.text else 'N/A'
            costs = extract_construction_costs(building)
            
            if costs:
                for ui in costs:
                    writer.writerow([building.guid, name, ui.text(), ui.value])
            else:
                writer.writerow([building.guid, name, 'Free', 0])
    
    print(f"Exported to: {output_path}")

# Export factory building costs
if "Production" in assets.templates.elements:
    export_costs_to_csv("Production", "factory_costs.csv")

Exported to: results\example\factory_costs.csv


## Next Steps

- `06_backtrack_effects.ipynb` - Find items that reduce construction costs
- `03_iterate_buildings.ipynb` - Learn more about building properties